# AI-Based Resume Screening System — Experiments & Pipeline Demo

This notebook provides a step-by-step interactive walkthrough of each component in the screening pipeline:
1. **Dataset Loading**: Job Requirements & Candidate Resumes
2. **Text Extraction**: PDF/DOCX parsing via PyMuPDF
3. **NLP Preprocessing**: Lowercasing, noise reduction, preserving tech keywords
4. **Transformer Embeddings**: Vector representation with `all-MiniLM-L6-v2`
5. **Similarity Matching**: Cosine similarity calculation
6. **Candidate Ranking**: Sorting and generating leaderboard
7. **Explainable AI (XAI)**: Skill gap analysis (matched vs missing skills)

In [ ]:
import os
import sys
import pandas as pd

# Ensure project root is in python path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data_loader import load_job_requirements, list_resume_files
from src.text_extraction import extract_text, extract_metadata
from src.preprocessing import clean_text
from src.embeddings import get_embedding_model, generate_embeddings
from src.matching import compute_cosine_similarity, compute_hybrid_score
from src.explainability import explain_candidate_match
from src.ranking import rank_candidates

print("All modules imported successfully!")

## Step 1: Load Job Requirements Dataset

In [ ]:
jobs_df = load_job_requirements("../dataset/job_requirements.csv")
print(f"Loaded {len(jobs_df)} job positions:")
display(jobs_df[["job_id", "job_title", "category", "experience_years", "required_skills"]])

## Step 2: Extract Text from Resumes (PyMuPDF)

In [ ]:
sample_files = list_resume_files("../dataset/resumes")
print(f"Found {len(sample_files)} resumes in dataset/resumes:")
for f in sample_files:
    print(f" - {f['filename']}")

# Extract first resume
first_resume = sample_files[0]
resume_text = extract_text(first_resume["filepath"], first_resume["filename"])
metadata = extract_metadata(resume_text, first_resume["filename"])

print(f"\nCandidate Name: {metadata['candidate_name']}")
print(f"Extracted text preview (first 300 chars):\n{resume_text[:300]}...")

## Step 3: NLP Preprocessing

In [ ]:
cleaned_resume = clean_text(resume_text)
print("Raw snippet:    ", resume_text[:120].replace('\n', ' '))
print("Cleaned snippet:", cleaned_resume[:120])

## Step 4: Generate Sentence Transformer Embeddings

In [ ]:
model = get_embedding_model("all-MiniLM-L6-v2")

# Select ML Engineer as target job
target_job = jobs_df.iloc[0].to_dict()
cleaned_jd = clean_text(target_job["full_job_text"])

jd_embedding = generate_embeddings(cleaned_jd, model=model)
resume_embedding = generate_embeddings(cleaned_resume, model=model)

print(f"JD Embedding Shape:     {jd_embedding.shape}")
print(f"Resume Embedding Shape: {resume_embedding.shape}")

## Step 5: Compute Cosine Similarity

In [ ]:
similarity = compute_cosine_similarity(jd_embedding, resume_embedding)[0]
print(f"Target Job: {target_job['job_title']}")
print(f"Candidate:  {metadata['candidate_name']}")
print(f"Semantic Cosine Similarity: {similarity * 100:.2f}%")

## Step 6 & 7: Rank All Candidates & Generate Explainable AI (XAI) Report

In [ ]:
candidates = []
for f in sample_files:
    t = extract_text(f["filepath"], f["filename"])
    m = extract_metadata(t, f["filename"])
    candidates.append({
        "filename": f["filename"],
        "raw_text": t,
        "candidate_name": m["candidate_name"],
        "email": m["email"],
        "phone": m["phone"]
    })

results, leaderboard = rank_candidates(candidates, target_job, use_hybrid=True, model=model)
display(leaderboard)

print("\n=== TOP CANDIDATE XAI BREAKDOWN ===")
top = results[0]
print(f"Candidate: {top['candidate_name']} ({top['final_score']}%)")
print(f"✓ Matched Skills ({len(top['matched_skills'])}): {', '.join(top['matched_skills'])}")
print(f"✗ Missing Skills ({len(top['missing_skills'])}): {', '.join(top['missing_skills'])}")
print(f"Experience Fit: {top['experience_status']}")
print(f"Justification: {top['summary']}")